# Etap 2 — Fine-tuning modelu

Fine-tuning XLM-RoBERTa (multi-label, 23 techniki perswazji) na danych z Etapu 1, z selekcją checkpointu na **polskim** dev.

- **Model:** `xlm-roberta-large` — zmiana na `base` to jedna linia `MODEL_NAME`
- **Loss:** `BCEWithLogitsLoss` z `pos_weight` z Etapu 1
- **Precyzja:** `bf16` — XLM-R + fp16 powoduje NaN
- **Checkpoint:** selekcja wg `f1_micro` na polskim dev (język docelowy bota)
- **`SMOKE_TEST`:** uruchom najpierw, by sprawdzić pipeline (500 próbek, 1 epoka)

In [ ]:
import sys
sys.modules["torchvision"] = None  # zapobiega konfliktowi VideoReader z datasets

from google.colab import drive
drive.mount("/content/drive")

!pip install -q -U transformers datasets accelerate scikit-learn

## 1. Konfiguracja

In [ ]:
from pathlib import Path
import math

# --- Ścieżki (zgodne z Etapem 1) ---
PROJECT_DIR = Path("/content/drive/MyDrive/Uczenie Maszynowe/Project")
DATA_DIR    = PROJECT_DIR / "Data"
PROC_DIR    = DATA_DIR / "processed"
MODELS_DIR  = PROJECT_DIR / "Models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROC_DIR / "semeval_multilang"
LABELS_PATH  = PROC_DIR / "labels.txt"
POSW_PATH    = PROC_DIR / "pos_weight.npy"

# --- Model ---
MODEL_NAME  = "xlm-roberta-large"   # base ⇄ large = ta jedna linia
TARGET_LANG = "po"
MAX_LENGTH  = 256

# --- Hiperparametry ---
LR              = 2e-5
TRAIN_BS        = 16
EVAL_BS         = 32
EPOCHS          = 10
WEIGHT_DECAY    = 0.01
WARMUP_FRAC     = 0.06
MAX_GRAD_NORM   = 1.0
PATIENCE        = 3
SEED            = 42

OUTPUT_DIR  = "/content/checkpoints"  # lokalnie (szybko); najlepszy model → Drive niżej
BEST_DIR    = MODELS_DIR / f"{MODEL_NAME.split('/')[-1]}_semeval_pl"

SMOKE_TEST  = False   # True = 500 próbek + 1 epoka (test pipeline)
print("Model:", MODEL_NAME, "| target:", TARGET_LANG, "| smoke:", SMOKE_TEST)

## 2. Dane + tokenizacja

In [ ]:
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer

# klasy (źródło prawdy z Etapu 1) i wagi do loss
LABELS = LABELS_PATH.read_text(encoding="utf-8").splitlines()
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABELS)
pos_weight_np = np.load(POSW_PATH)
assert NUM_LABELS == 23 == len(pos_weight_np)

ds = load_from_disk(str(DATASET_PATH))
print(ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_tok = ds.map(tokenize, batched=True)

# zbiór treningowy = wszystkie języki; eval = TYLKO polski dev (selekcja pod bota)
train_ds = ds_tok["train"]
eval_ds  = ds_tok["validation"].filter(lambda x: x["lang"] == TARGET_LANG)

if SMOKE_TEST:
    train_ds = train_ds.shuffle(seed=SEED).select(range(500))
    EPOCHS = 1
    print(">>> SMOKE TEST: 500 próbek, 1 epoka")

print(f"train: {len(train_ds)} | eval ({TARGET_LANG} dev): {len(eval_ds)}")


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 19901
    })
    validation: Dataset({
        features: ['text', 'labels', 'lang', 'article_id', 'paragraph_id'],
        num_rows: 6456
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

train: 19901 | eval (po dev): 800


## 3. Data collator (multi-label)

Dynamiczny padding `input_ids`/`attention_mask` + etykiety jako tensor `float32` o kształcie `(batch, 23)`.

In [ ]:
import torch
from dataclasses import dataclass

@dataclass
class MultiLabelCollator:
    tokenizer: object
    def __call__(self, features):
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True, return_tensors="pt",
        )
        batch["labels"] = labels
        return batch

collator = MultiLabelCollator(tokenizer)


## 4. Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

## 5. Loss (pos_weight) + metryki

In [ ]:
from torch import nn
from transformers import Trainer
from sklearn.metrics import f1_score

class MultiLabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight  # tensor (NUM_LABELS,)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= 0.5).astype(int)     # próg 0.5 tylko do monitoringu
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

pos_weight_t = torch.tensor(pos_weight_np, dtype=torch.float32)


## 6. TrainingArguments + trening

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

steps_per_epoch = math.ceil(len(train_ds) / TRAIN_BS)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(WARMUP_FRAC * total_steps)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    max_grad_norm=MAX_GRAD_NORM,
    bf16=True,                            # bf16, NIE fp16 — XLM-R + fp16 daje NaN
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = MultiLabelTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_t,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

trainer.train()

## 7. Zapis najlepszego modelu na Drive

`load_best_model_at_end=True` → `trainer.model` zawiera najlepszy checkpoint wg polskiego dev micro-F1.

In [ ]:
import json

trainer.save_model(str(BEST_DIR))          # model + config
tokenizer.save_pretrained(str(BEST_DIR))   # tokenizer

# zapisz też listę klas obok modelu (żeby bot/Etap 3 nie zależał od innych plików)
(BEST_DIR / "labels.txt").write_text("\n".join(LABELS), encoding="utf-8")

info = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": NUM_LABELS,
    "best_metric_f1_micro_po_dev": trainer.state.best_metric,
    "epochs_planned": EPOCHS,
}
(BEST_DIR / "training_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))
print("Zapisano model do:", BEST_DIR)
print(info)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Zapisano model do: /content/drive/MyDrive/Uczenie Maszynowe/Project/Models/xlm-roberta-large_semeval_pl
{'model_name': 'xlm-roberta-large', 'max_length': 256, 'num_labels': 23, 'best_metric_f1_micro_po_dev': 0.4431946006749156, 'epochs_planned': 10}


## 8. Szybka ewaluacja (polski dev)

In [ ]:
metrics = trainer.evaluate()
print(f"Polski dev — micro-F1: {metrics['eval_f1_micro']:.4f} | macro-F1: {metrics['eval_f1_macro']:.4f}")


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.074875,0.402370,9,0.443195,0.322865


Polski dev — micro-F1: 0.4432 | macro-F1: 0.3229


---
## Co dalej — Etap 3

`3_analiza.ipynb` wczyta model i dostroją progi per-klasa na polskim dev (`thresholds.npy`) — to daje główny skok micro-F1 ponad próg 0.5. Progi + model = wszystko, czego potrzebuje bot.